# Agentic RAG with Tool Calling

This notebook is written as a study guide. It follows the lesson flow step by step and adds explanations so you can understand **what each cell is doing** and **why it matters**.

Flow:
1. Load and index documentation
2. Build traditional RAG
3. Convert it to agentic RAG with a tool
4. Add structured output


In [ ]:
# Install required packages for the notebook.
# Run this cell once. If packages are already installed, it will be quick.
%pip install -q openai gitsource minsearch pydantic


## Imports and OpenAI client

We import the libraries used in the lesson.

- `OpenAI` lets us call the model.
- `GithubRepositoryDataReader` reads docs from GitHub.
- `chunk_documents` splits long docs into smaller pieces for retrieval.
- `AppendableIndex` is the search index.
- `BaseModel` and `Field` are for structured output later.


In [ ]:
from openai import OpenAI
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex
from pydantic import BaseModel, Field
from typing import Literal
import json
from pprint import pprint

# Create OpenAI client.
# Make sure your OPENAI_API_KEY is available in the environment.
openai_client = OpenAI()


## Load documentation from GitHub

This is the knowledge base for our assistant. The lesson uses the Evidently docs repository.

Why we do this:
- The model should answer using docs, not only its internal knowledge.
- We need searchable text chunks for retrieval.


In [ ]:
# Read markdown documentation files from the Evidently docs repo.
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()

# Parse files into structured document objects.
parsed_docs = [doc.parse() for doc in files]

# Split long documents into overlapping chunks.
# size=3000 means each chunk is about 3000 characters.
# step=1500 means chunks overlap, which helps preserve context.
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

print('Files loaded:', len(files))
print('Chunks created:', len(chunked_docs))


## Build the search index

The search index is what powers retrieval.

- `text_fields` are used for searching by meaning/text similarity.
- `keyword_fields` are useful for exact-match fields such as filenames.


In [ ]:
# Build an index over the chunked documentation.
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)

index.fit(chunked_docs)

print('Index built successfully.')


## Define the search function

This is the core retrieval function.

Important idea:
- In **traditional RAG**, our code calls this directly.
- In **agentic RAG**, the model asks our app to call this tool.


In [ ]:
def search(query):
    # Search the documentation index and return up to 5 results.
    results = index.search(
        query=query,
        num_results=5
    )
    return results


## Traditional RAG setup

Traditional RAG means: our code controls the whole flow.

Flow:
1. Take user question
2. Search docs with that exact question
3. Put retrieved results into the prompt
4. Ask the model to answer from the provided context

This is simple, but it has a weakness: if the query is poor or misspelled, retrieval quality drops.


In [ ]:
# These are the system instructions for the traditional RAG step.
# The model is told to answer ONLY using the supplied context.
RAG_INSTRUCTIONS = '''
You're a documentation assistant.
Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer is not in the CONTEXT, say so.
'''.strip()

# The prompt template contains two parts:
# - QUESTION: the user's question
# - CONTEXT: retrieved docs as JSON text
RAG_PROMPT_TEMPLATE = '''
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
'''.strip()

def build_prompt(question, search_results):
    # Convert the retrieved results into a pretty JSON string.
    # This becomes the context shown to the model.
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(question=question, context=context)


## Traditional RAG example

We intentionally use a misspelled question here: `dahsbord` instead of `dashboard`.

This shows why traditional RAG can fail: the search step uses the raw question exactly as written.


In [ ]:
# Example question with a typo.
question = 'How do I create a dahsbord in Evidently?'

# Traditional RAG searches immediately using the raw question.
search_results = search(question)

# Build prompt with question + retrieved context.
user_prompt = build_prompt(question, search_results)

# In traditional RAG, the system message contains the rules,
# and the user message contains both question and retrieved docs.
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

print('Traditional RAG answer:')
print()
print(response.output_text)


## Move to agentic RAG

Now we change the architecture.

Instead of always searching first, we let the model decide:
- whether to search,
- what query to use,
- and when it has enough information.

That decision-making step is what makes it **agentic**.


In [ ]:
# These are the system instructions for the agentic version.
# Notice that we do NOT give context yet.
# We only define the model's role and constraints.
instructions = '''
You're a documentation assistant.

Answer the user question using the documentation knowledge base.

Use only facts from the knowledge base when answering.
IMPORTANT: if you cannot find the answer, inform the user.
'''.strip()

# This is the tool schema shown to the model.
# It tells the model that a function named 'search' exists,
# what it does, and what arguments it accepts.
search_tool = {
    'type': 'function',
    'name': 'search',
    'description': 'Search the documentation database for relevant results based on a query string.',
    'parameters': {
        'type': 'object',
        'properties': {
            'query': {
                'type': 'string',
                'description': 'The search query to look up in the index'
            }
        },
        'required': ['query']
    }
}


## Compare token count without tools

First, ask the model the question **without** giving it the search tool.

The lesson uses this to show how token count changes once tool definitions are added.


In [ ]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response_no_tools = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

print('Input tokens without tools:', response_no_tools.usage.input_tokens)


## First agentic call with tool available

Now we pass the tool schema to the model.

This does **not** run the search automatically. It only tells the model:
"You may call this tool if you need it."


In [ ]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

print('Input tokens with tool schema:', response.usage.input_tokens)
print()
print('Raw response.output:')
pprint(response.output)


## Inspect the tool call

At this step, the model often decides to call `search`.

Key observation:
- The model may improve the query before searching.
- For example, it can fix `dahsbord` to `dashboard`.

This is the main improvement over traditional RAG.


In [ ]:
# The first item in response.output is typically the tool call request.
tool_call = response.output[0]

print('Tool call object:')
pprint(tool_call)
print()
print('Tool call arguments (raw JSON string):')
print(tool_call.arguments)


## Parse tool arguments and execute the real Python function

This part is important conceptually.

The LLM does **not** execute Python code itself. Instead:
1. The model requests a tool call.
2. Our app reads the requested arguments.
3. Our Python code runs the real function.
4. We pass the result back to the model.


In [ ]:
# Convert tool arguments from JSON string to Python dictionary.
arguments = json.loads(tool_call.arguments)

print('Parsed tool arguments:')
pprint(arguments)

# Run the actual Python search function with those arguments.
search_results = search(**arguments)

print()
print('Number of search results:', len(search_results))
print()
print('First search result preview:')
pprint(search_results[:1])


## Append tool call and tool output to the conversation

We now extend the conversation history with:
- the tool call requested by the model, and
- the tool result produced by our app.

This gives the model both the request and the evidence it asked for.


In [ ]:
# Keep the tool call in the conversation history.
messages.append(tool_call)

# Return the search results in the expected tool-output format.
call_output = {
    'type': 'function_call_output',
    'call_id': tool_call.call_id,
    'output': json.dumps(search_results),
}

messages.append(call_output)

print('Total messages in conversation:', len(messages))


## Final agentic answer

Now the model has the retrieved documentation results and can generate a grounded answer.


In [ ]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

print('Input tokens after sending search results:', response.usage.input_tokens)
print()
print('Final agentic RAG answer:')
print()
print(response.output_text)


## Add structured output

So far, the agent returns plain text.

Now we define a Pydantic schema so the final answer becomes a validated Python object with fields like:
- `answer`
- `found_answer`
- `confidence`
- `answer_type`
- `followup_questions`


In [ ]:
class RAGResponse(BaseModel):
    # Main answer in markdown/text form.
    answer: str = Field(description="The main answer to the user's question in markdown")

    # Whether the model believes the docs contained enough evidence.
    found_answer: bool = Field(description="True if relevant information was found in the knowledge base")

    # A score from 0.0 to 1.0 indicating confidence.
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how well the answer is supported")

    # Why the model chose that confidence level.
    confidence_explanation: str = Field(description="Explanation about the confidence level")

    # A category that tells us what kind of answer this is.
    answer_type: Literal['how-to', 'explanation', 'troubleshooting', 'comparison', 'reference'] = Field(description="Category of the answer")

    # Useful next questions the user could ask.
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user may ask next")


## Request structured output

We now call `responses.parse(...)` instead of `responses.create(...)`.

This asks the API to return output that fits the `RAGResponse` schema.


In [ ]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

print('Input tokens with structured schema:', response.usage.input_tokens)


## Read the structured result

The parsed object is available in `response.output_parsed`.

This is much easier to work with in code than unstructured plain text.


In [ ]:
rag_response = response.output_parsed

print('Structured answer:')
print()
print(rag_response.answer)

print()
print('found_answer:', rag_response.found_answer)
print('confidence:', rag_response.confidence)
print('confidence_explanation:', rag_response.confidence_explanation)
print('answer_type:', rag_response.answer_type)
print('followup_questions:')
for q in rag_response.followup_questions:
    print('-', q)


## Optional helper function

This helper wraps the agentic flow into one reusable function.

Use it after you understand the manual steps above. That manual flow is the most important part for learning.


In [ ]:
def ask_agentic_docs(question, model='gpt-4o-mini'):
    # Start a new conversation with system instructions + user question.
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": question}
    ]

    # First call: let the model decide whether it wants to use the search tool.
    first_response = openai_client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool],
    )

    # If the model returned plain text and did not use a tool, return that text.
    if not first_response.output:
        return first_response.output_text

    first_item = first_response.output[0]

    if getattr(first_item, 'type', None) != 'function_call':
        return first_response.output_text

    # Otherwise, extract tool arguments and execute the real Python function.
    tool_call = first_item
    arguments = json.loads(tool_call.arguments)
    results = search(**arguments)

    # Add tool call and tool result back into conversation history.
    messages.append(tool_call)
    messages.append({
        'type': 'function_call_output',
        'call_id': tool_call.call_id,
        'output': json.dumps(results),
    })

    # Final call: model now has tool evidence and can answer.
    final_response = openai_client.responses.create(
        model=model,
        input=messages,
        tools=[search_tool],
    )

    return final_response.output_text


## Try the helper

Once the flow makes sense, use the helper for shorter experiments.


In [ ]:
print(ask_agentic_docs('How do I create a dahsbord in Evidently?'))
